# 🧠 Tutor Virtual Pedagógico de Machine Learning

### 1. Documentación
- Carga de documentos PDF.
- Extracción y fragmentación del texto (Chunks).

### 2. Base de datos vectorial (RAG)
- Generación de embeddings 
- Almacenar los chunks en ChromaDB

### 3. Recuperación de información (Retriever)
- Búsqueda semántica mediante Retriever.
- Recuperación de fragmentos relevantes.

## 4. System prompt
Define el rol del tutor, las reglas de uso de la herramienta y el formato
de respuesta.

### 5. Construccion del agente con LangGraph
- Definir el modelo.
- Crear Nodos.
- Herramienta `buscar_documentacion`
- Crear el Agente
- Conecta el agente

## 6. Consulta al tutor
- Normaliza el contenido de la respuesta
- Consultar el Agente con `consultar_tutor`


## 7. Evaluar el Agente

In [1]:
import os
import time
import uuid

from dotenv import load_dotenv
from pathlib import Path
from typing import Literal

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.tools import tool
from langchain_core.messages import (
    SystemMessage,
    ToolMessage,
    HumanMessage
)

from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)

from langgraph.checkpoint.memory import InMemorySaver


C:\Users\rubye\AppData\Local\Temp\ipykernel_18312\2383383.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Configuración general

In [ ]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = "gemini-3.5-flash-lite"

DOCUMENTS_PATH = Path("data/documents")
CHROMA_PATH = "data/chroma"
COLLECTION_NAME = "tutor_machine_learning"

checkpointer = InMemorySaver()
thread_id = str(uuid.uuid4())
app_tutor = None
total_calls = 0


## 1. Documentación 
- Carga de documentos PDF
- Extracción y fragmentación del texto (Chunks)


In [3]:
def cargar_documentos():
    documentos = []
    if not DOCUMENTS_PATH.exists():
        return documentos
    archivos_pdf = list(DOCUMENTS_PATH.glob("*.pdf"))
    for archivo in archivos_pdf:
        loader = PyPDFLoader(str(archivo))
        paginas = loader.load()
        documentos.extend(paginas)
    return documentos


In [4]:
def crear_chunks():
    documentos = cargar_documentos()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documentos)
    return chunks

## 2. Base de datos vectorial (RAG)
- Generación de embeddings 
- Almacenar los chunks en ChromaDB

In [5]:
def obtener_vectorstore():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )
    # Si la coleccion esta vacia, indexar los documentos.
    if vectorstore._collection.count() == 0:
        chunks = crear_chunks()
        if not chunks:
            raise ValueError("No se encontraron documentos PDF en data/documents.")
        vectorstore.add_documents(documents=chunks)
    return vectorstore


## 3. Recuperación de información (Retriever)
- Búsqueda semántica mediante Retriever.
- Recuperación de fragmentos relevantes.

In [6]:
def obtener_retriever():
    vectorstore = obtener_vectorstore()
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )
    return retriever


In [7]:
retriever = obtener_retriever()

# Herramienta
@tool
def buscar_documentacion(query: str) -> str:
    """Busca informacion relevante en la documentacion."""
    docs = retriever.invoke(query)
    if not docs:
        return "NO_INFO: No se encontro informacion relevante en los apuntes."
    resultados = []
    for doc in docs:
        fuente = doc.metadata.get("source", "Desconocido")
        pagina = doc.metadata.get("page", 0) + 1
        resultados.append(
            f"[Documento: {fuente} | Pagina: {pagina}]\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(resultados)


tools = [buscar_documentacion]
tools_by_name = {
    t.name: t for t in tools
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 4. System prompt

Define el rol del tutor, las reglas de uso de la herramienta y el formato
de respuesta.


In [8]:
SYSTEM_PROMPT = """
Eres un Tutor Virtual Pedagogico de Machine Learning.
Responde EXCLUSIVAMENTE con la documentacion del curso.

REGLAS:
1. Invoca SIEMPRE 'buscar_documentacion' antes de responder.
2. Usa solo la info recuperada. Si no hay info, responde:
   "Lo siento, ese tema no esta en los materiales del curso."
3. Incluye siempre documento y pagina de cada cita.
4. Responde en espanol, claro, pedagogico y bien estructurado.
"""


## 5. Construccion del agente con LangGraph

- Definir el modelo.
- Crear Nodos.
- Herramienta `buscar_documentacion`
- Crear el Agente
- Conecta el agente

In [ ]:
def construir_agente(api_key):
    llm = ChatGoogleGenerativeAI(
        model=MODEL,
        google_api_key=api_key,
        max_output_tokens=1024
    )
    llm_with_tools = llm.bind_tools(tools)

    # Nodo del modelo
    def llm_call(state: MessagesState):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    # Nodo de herramientas
    def tool_node(state: MessagesState):
        last_message = state["messages"][-1]
        results = []
        for tool_call in last_message.tool_calls:
            tool_func = tools_by_name[tool_call["name"]]
            observation = tool_func.invoke(tool_call["args"])
            results.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
        return {"messages": results}

    # Router
    def should_continue(state: MessagesState) -> Literal["tool_node", END]:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tool_node"
        return END

    # Construccion del grafo
    grafo = StateGraph(MessagesState)
    grafo.add_node("llm_call", llm_call)
    grafo.add_node("tool_node", tool_node)
    grafo.add_edge(START, "llm_call")   
    grafo.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    grafo.add_edge("tool_node", "llm_call")

    # Memory
    app_tutor = grafo.compile(checkpointer=checkpointer)
    return app_tutor


In [ ]:
app_tutor = construir_agente(API_KEY)

Agente conectado correctamente.


## 6. Consultar el Agente

- Normaliza el contenido de la respuesta
- Consultar el Agente con `consultar_tutor`


In [11]:
# Normaliza el contenido de la respuesta
def obtener_texto_limpio(content):
    """Extrae texto limpio"""
    if isinstance(content, list):
        textos = [b.get("text", "") for b in content if isinstance(b, dict) and "text" in b]
        return "\n".join(textos)
    return str(content)


In [12]:
def consultar_tutor(pregunta: str):
    global total_calls
    inicio = time.perf_counter()
    resultado = app_tutor.invoke(
        {
            "messages": [HumanMessage(content=pregunta)]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )
    respuesta_raw = resultado["messages"][-1].content
    respuesta_limpia = obtener_texto_limpio(respuesta_raw)
    tiempo_respuesta = time.perf_counter() - inicio
    total_calls += 1
    return respuesta_limpia, tiempo_respuesta

## 7. Evaluar el Agente

In [ ]:
preguntas_evaluacion = [
    #"¿Qué es Machine Learning y cuáles son sus tipos principales?",
    #"¿Qué es Deep Learning y cuales son sus principios?",
    #"¿Cómo funciona el algoritmo K-Means?",
    #"¿Podrías darme un ejemplo rápido en código Python del algoritmo que me acabas de explicar?",
    "¿Quién fue el primer presidente de los Estados Unidos?",
]

for i, pregunta in enumerate(preguntas_evaluacion, 1):
    print("=" * 80)
    print(f"📌 Pregunta: {i} - {pregunta}")
    resp, t = consultar_tutor(pregunta)
    print(f"📌 Respuesta:")
    print(resp)
    print(f"\n⏱️ Tiempo: {t:.2f}s")


📌 Pregunta: 1 - ¿Quién fue el primer presidente de los Estados Unidos?---------------
📌 Respuesta:
Lo siento, ese tema no esta en los materiales del curso.

⏱️ Tiempo: 1.43s


: 